In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

In [24]:
df = pd.read_csv("knn_telecom.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Age           150 non-null    int64 
 1   Gender        150 non-null    object
 2   PlanType      150 non-null    object
 3   Tenure        150 non-null    int64 
 4   MonthlyUsage  150 non-null    int64 
 5   Churn         150 non-null    int64 
dtypes: int64(4), object(2)
memory usage: 7.2+ KB


In [25]:
df = df.loc[:, ~df.columns.str.lower().str.contains("id")]

In [26]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].mean())    

In [35]:
X = df.drop("Churn", axis=1)
Y = df["Churn"]
X,Y
Y = np.ravel(Y)

In [36]:
numeric_features = X.select_dtypes(include=["int64","float64"]).columns
categorial_features = X.select_dtypes(include=["object"]).columns 

In [37]:
preprocessor =  ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorial_features)
    ]
)

In [38]:
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

k_values = range(1,16)
cv_scores = []


for k in k_values: 
    model = Pipeline([
        ("preprocessing", preprocessor),
        ("knn",KNeighborsClassifier(n_neighbors=k))
        
    ])
    scores = cross_val_score(model, X, Y , cv=skf, scoring = "accuracy")
    cv_scores.append(scores.mean())
    
    
    

TypeError: iteration over a 0-d array

In [ ]:
for k, score in zip(k_values, cv_scores):
    print(f"K = {k} | CV Accuracy = {round(score,4)}")
    
    
best_k = k_values[np.argmax(cv_scores)]
print("\nBest K Selected:",best_k)    

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(k_values,cv_scores,marker='0')
plt.xlabel("K value")
plt.ylabel("Cross Validation Accuracy")
plt.title("Selecting Optimal K")
plt.xticks(k_values)
plt.grid(True)
plt.show()

In [ ]:
final_model = Pipeline([
    ("preprocessing", preprocessor),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))
])

final_model.fit(X,Y)

In [ ]:
user_data = {
    "Age":40,
    "Gender":"Male",
    "PlanType": "Premium",
    "Tenure": 12,
    "MonthlyUsage":350
    }

user_df = pd.DataFrame([user_data])


prediction = final_model.predict(user_df)[0]
probability = final_model.predict_proba(user_df)[0]


print("\n user Input:")
print(user_df)

print("\nPredicted Churn:",prediction)
print("\nPrediction Probability: ",probability)